In [1]:
!pip install transformers accelerate hf_transfer peft -Uqq

In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'LGAI-EXAONE/EXAONE-4.0-1.2B'
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 332/332 [00:11<00:00, 29.23it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [3]:
prompt = '파이썬이 뭐냐'
chosen = '파이썬은 배우기 쉽고, 강력한 프로그래밍 언어입니다'
rejected = '파이썬은 뱀의 일종이다'

In [4]:
def get_logprob(model, tokenizer, prompt, response):
    full_text = prompt + response
    inputs = tokenizer(full_text, return_tensors='pt').to(model.device)
    prompt_len = len(tokenizer(prompt, return_tensors='pt')['input_ids'][0])

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        log_probs = F.log_softmax(logits, dim=-1)

        token_log_probs = log_probs[:, :-1].gather(
            index=inputs['input_ids'][:, 1:].unsqueeze(-1),
            dim=-1
        ).squeeze(-1)

        response_log_probs = token_log_probs[:, prompt_len-1:]
        total = response_log_probs.sum()
    return total.item()

chosen_prob = get_logprob(model, tokenizer, prompt, chosen)
rejected_prob = get_logprob(model, tokenizer, prompt, rejected)
print(f'선호 답변 생성 확률: {chosen_prob}, 비선호 답변 생성 확률: {rejected_prob}')

선호 답변 생성 확률: -88.5, 비선호 답변 생성 확률: -53.25


In [5]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    logits = beta * ((policy_chosen - policy_rejected) - (ref_chosen - ref_rejected))
    loss = -F.logsigmoid(torch.tensor(logits))
    return loss.item()